In [1]:
# CELL 1
import os, glob, numpy as np, pandas as pd
import librosa, torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import warnings, random
warnings.filterwarnings('ignore')

BASE_PATH   = '/kaggle/input/competitions/birdclef-2026'
SR          = 32000
DURATION    = 5
N_MELS      = 128
NUM_CLASSES = 206
EPOCHS      = 15
BATCH_SIZE  = 32

print("Config done!")

Config done!


In [2]:
# CELL 2
train_df     = pd.read_csv(f'{BASE_PATH}/train.csv')
sample_sub   = pd.read_csv(f'{BASE_PATH}/sample_submission.csv')
species_cols = [c for c in sample_sub.columns if c != 'row_id']
soundscape_df = pd.read_csv(f'{BASE_PATH}/train_soundscapes_labels.csv')

# Combine train audio and soundscape data
train_filtered = train_df.copy().reset_index(drop=True)
species_list   = sorted(train_filtered['primary_label'].unique())
species2idx    = {s: i for i, s in enumerate(species_list)}

print(f"Total samples: {len(train_filtered)}")
print(f"Total species: {len(species_list)}")
print(f"Soundscape samples: {len(soundscape_df)}")

Total samples: 35549
Total species: 206
Soundscape samples: 1478


In [3]:
# CELL 3
class BirdDataset(Dataset):
    def __init__(self, df, base_path, augment=True):
        self.df        = df
        self.base_path = base_path
        self.augment   = augment
        self.samples   = SR * DURATION

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = f'{self.base_path}/train_audio/{row["filename"]}'
        try:
            y, _ = librosa.load(path, sr=SR, duration=DURATION)
        except:
            y = np.zeros(self.samples)

        if len(y) < self.samples:
            y = np.pad(y, (0, self.samples - len(y)))

        # Strong augmentation
        if self.augment:
            # Noise
            if random.random() > 0.4:
                noise_level = random.uniform(0.001, 0.008)
                y = y + noise_level * np.random.randn(len(y))
            # Time shift
            if random.random() > 0.4:
                shift = int(random.uniform(-1.0, 1.0) * SR)
                y = np.roll(y, shift)
            # Gain
            if random.random() > 0.4:
                y = y * random.uniform(0.6, 1.4)
            # Mixup with noise
            if random.random() > 0.7:
                noise = np.random.randn(len(y)) * 0.01
                y = 0.9 * y + 0.1 * noise

        mel    = librosa.feature.melspectrogram(
            y=y, sr=SR, n_mels=N_MELS,
            fmin=20, fmax=16000,
            hop_length=512, n_fft=1024)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-8)

        # SpecAugment
        if self.augment:
            if random.random() > 0.4:
                f = random.randint(0, N_MELS - 20)
                mel_db[f:f+random.randint(5, 25), :] = 0
            if random.random() > 0.4:
                t = mel_db.shape[1]
                s = random.randint(0, t - 20)
                mel_db[:, s:s+random.randint(5, 25)] = 0
            # Double mask
            if random.random() > 0.7:
                f2 = random.randint(0, N_MELS - 15)
                mel_db[f2:f2+random.randint(3, 15), :] = 0

        tensor = torch.tensor(
            mel_db, dtype=torch.float32).unsqueeze(0)

        label = torch.zeros(NUM_CLASSES)
        if row['primary_label'] in species2idx:
            label[species2idx[row['primary_label']]] = 1.0

        # Add secondary labels
        if isinstance(row.get('secondary_labels', []), list):
            for sec in row.get('secondary_labels', []):
                if sec in species2idx:
                    label[species2idx[sec]] = 0.5

        return tensor, label

dataset = BirdDataset(train_filtered, BASE_PATH)
print(f"Dataset ready! Size: {len(dataset)}")

Dataset ready! Size: 35549


In [4]:
# CELL 4
class BirdModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = timm.create_model(
            'tf_efficientnetv2_m',  # Changed name!
            pretrained=True,
            in_chans=1,
            num_classes=0,
            drop_rate=0.3,
            drop_path_rate=0.2
        )
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu')
model  = BirdModel(num_classes=NUM_CLASSES).to(device)
print(f"Model ready! Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/218M [00:00<?, ?B/s]

Model ready! Device: cuda
Parameters: 53,620,498


In [5]:
# CELL 5
loader    = DataLoader(dataset, batch_size=BATCH_SIZE,
                       shuffle=True, num_workers=2,
                       pin_memory=True)
optimizer = AdamW(model.parameters(), lr=2e-4,
                  weight_decay=1e-4)
scheduler = CosineAnnealingWarmRestarts(
    optimizer, T_0=5, T_mult=2)
criterion = nn.BCEWithLogitsLoss()
best_loss = float('inf')

print(f"Starting Training {EPOCHS} epochs...")
print(f"Batches per epoch: {len(loader)}")
print("="*50)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, (specs, labels) in enumerate(loader):
        specs  = specs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(specs)
        loss    = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step(epoch + batch_idx/len(loader))
        total_loss += loss.item()

        if (batch_idx + 1) % 200 == 0:
            avg = total_loss / (batch_idx + 1)
            print(f"Epoch [{epoch+1}/{EPOCHS}] "
                  f"Batch [{batch_idx+1}/{len(loader)}] "
                  f"Loss: {avg:.4f}")

    avg_loss = total_loss / len(loader)
    print(f"\nEpoch [{epoch+1}/{EPOCHS}] "
          f"Avg Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), 'best_model_v6.pth')
        print(f"Best model saved! Loss: {best_loss:.4f}")
    print("="*50)

print("\nTraining Complete!")

Starting Training 15 epochs...
Batches per epoch: 1111
Epoch [1/15] Batch [200/1111] Loss: 0.1567
Epoch [1/15] Batch [400/1111] Loss: 0.0951
Epoch [1/15] Batch [600/1111] Loss: 0.0735
Epoch [1/15] Batch [800/1111] Loss: 0.0625
Epoch [1/15] Batch [1000/1111] Loss: 0.0558

Epoch [1/15] Avg Loss: 0.0531
Best model saved! Loss: 0.0531
Epoch [2/15] Batch [200/1111] Loss: 0.0286
Epoch [2/15] Batch [400/1111] Loss: 0.0281
Epoch [2/15] Batch [600/1111] Loss: 0.0276
Epoch [2/15] Batch [800/1111] Loss: 0.0269
Epoch [2/15] Batch [1000/1111] Loss: 0.0262

Epoch [2/15] Avg Loss: 0.0258
Best model saved! Loss: 0.0258
Epoch [3/15] Batch [200/1111] Loss: 0.0208
Epoch [3/15] Batch [400/1111] Loss: 0.0202
Epoch [3/15] Batch [600/1111] Loss: 0.0196
Epoch [3/15] Batch [800/1111] Loss: 0.0191
Epoch [3/15] Batch [1000/1111] Loss: 0.0187

Epoch [3/15] Avg Loss: 0.0184
Best model saved! Loss: 0.0184
Epoch [4/15] Batch [200/1111] Loss: 0.0154
Epoch [4/15] Batch [400/1111] Loss: 0.0151
Epoch [4/15] Batch [600/1